# v22: 多scale 尤度加重アンサンブル(run_pf_lik_ensemble_scales 相当)を GPU で実験

**狙い**: 7.191 の `run_pf_lik_ensemble_scales` のアルゴリズム ―― 「PFを多シード→尤度で加重平均→複数の鋭さ(scale)で出す→selectorで選ぶ」 ――
を **v20 GPU PF(r0, パラメータは現状そのまま)** で実装し、scale と選択戦略、seed数を実験する。

## 検証する3点
1. **各 scale(温度)の精度**: 鋭い(scale小=最尤シードに賭ける) vs 緩い(scale大=hedge)。easy/hard 別。
2. **選択の天井 vs 到達可能**: oracle(坑井ごと最良scale=上限, 真値必要) と 固定scale/簡易ルールの差(=選択の壁)。
3. **seed数の効き**: 8/16/32/64 で pf_scale_8 の精度・安定性。多いほど正確か?(=安定性であって精度でない仮説の検証)

PFパラメータは触らない(r0/v20 のまま)。実行はユーザー。

In [7]:
import sys, os, glob, time, numpy as np
from collections import Counter
sys.path.insert(0, "../v21")
import importlib, v21_common as C; importlib.reload(C)

# ===== 設定(PFパラメータは r0 のまま)=====
SCALES   = (1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 20.0)   # 温度を広めにスイープ
SEEDS    = 128          # v20 の調整regime
CHUNK    = 100           # VRAMで調整(OOMなら下げる)
USE_PHYS = True
N_WELLS  = 773         # 検証サブセット(+ 硬い核20を必ず追加)

ns = C.load_v9_ns()
p  = C.r0_params(ns, use_phys=USE_PHYS, n_seeds=SEEDS)
wells = list(dict.fromkeys(C.list_train_wells()[:N_WELLS] + C.HARD_WELLS))
print("wells:", len(wells), "| device:", C.DEVICE, "| scales:", SCALES)

[v9] exec 中(FI/DI fit を含む)...
PF parameter sets:
  pf_1: suffix=_1, N=600, seeds=32, jump=(0.002, 5.0)
  pf_2: suffix=_2, N=600, seeds=32, jump=(0.0, 10.0)
  pf_3: suffix=_3, N=600, seeds=32, jump=(0.002, 5.0)
[v21] r0 params: r0 N=500 seeds=128 phys=on
wells: 773 | device: cuda | scales: (1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 20.0)


In [8]:
# ===== 入力構築 + 多scale GPU 走行 =====
inps, names, truth = [], [], {}
for wid in wells:
    x = C.build_inputs(wid, ns, p, split="train")
    if x is None or x["tvt_true"] is None: continue
    inps.append(x); names.append(wid); truth[wid] = x["tvt_true"]
t0 = time.perf_counter()
res = C.run_forward_scales(inps, p, SEEDS, C.DEVICE, 0, CHUNK, SCALES)
CACHE = {wid: d for wid, d in zip(names, res)}
print(f"done {len(CACHE)} wells in {time.perf_counter()-t0:.1f}s ({C.DEVICE})")
print("keys:", [k for k in next(iter(CACHE.values())) if k.startswith('pf_')])

done 773 wells in 485.2s (cuda)
keys: ['pf_scale_1', 'pf_scale_2', 'pf_scale_3', 'pf_scale_5', 'pf_scale_8', 'pf_scale_12', 'pf_scale_20', 'pf_ancc', 'pf_mean']


In [9]:
# ===== 評価1: 各 scale の easy/hard/all + oracle(坑井ごと最良scale)=====
def rmse(parts):
    e = np.concatenate(parts) if parts else np.array([np.nan]); e = e[np.isfinite(e)]
    return float(np.sqrt(np.mean(e**2))) if len(e) else float("nan")
def well_rmse(w, k): return rmse([CACHE[w][k]-truth[w]])
KEYS = [f"pf_scale_{s:g}" for s in SCALES] + ["pf_mean"]
EASY = [w for w in names if w not in C.HARD_WELLS]; HARD = [w for w in names if w in C.HARD_WELLS]
def grp(k, ws): return rmse([CACHE[w][k]-truth[w] for w in ws])

print(f"{'output':>12s} {'easy':>7s} {'hard':>7s} {'all':>7s}")
for k in KEYS:
    print(f"{k:>12s} {grp(k,EASY):7.2f} {grp(k,HARD):7.2f} {grp(k,names):7.2f}")

# oracle: 各坑井で最良scaleを選ぶ(到達不能の上限)
picks = {w: min(KEYS, key=lambda k: well_rmse(w, k)) for w in names}
oe = rmse([CACHE[w][picks[w]]-truth[w] for w in EASY])
oh = rmse([CACHE[w][picks[w]]-truth[w] for w in HARD])
oa = rmse([CACHE[w][picks[w]]-truth[w] for w in names])
print(f"{'ORACLE':>12s} {oe:7.2f} {oh:7.2f} {oa:7.2f}   ← 坑井ごと最良scale(上限)")
print("oracle pick分布:", dict(Counter(picks.values())))
print("\n★ 各scaleの最良(固定)と oracle の差 = 選択で取れる余地。easy/hard で傾向が違うか見る。")

      output    easy    hard     all
  pf_scale_1    9.80   25.57   10.50
  pf_scale_2    9.00   24.76    9.73
  pf_scale_3    8.55   24.09    9.27
  pf_scale_5    8.21   25.00    9.03
  pf_scale_8    8.08   25.89    8.98
 pf_scale_12    8.01   26.29    8.95
 pf_scale_20    7.99   26.62    8.96
     pf_mean    8.22   24.35    9.00
      ORACLE    7.09   20.73    7.74   ← 坑井ごと最良scale(上限)
oracle pick分布: {'pf_scale_1': 287, 'pf_scale_20': 239, 'pf_scale_2': 94, 'pf_scale_5': 36, 'pf_scale_12': 19, 'pf_scale_3': 60, 'pf_scale_8': 23, 'pf_mean': 15}

★ 各scaleの最良(固定)と oracle の差 = 選択で取れる余地。easy/hard で傾向が違うか見る。


In [10]:
# ===== 評価2: 到達可能な選択戦略 vs oracle(選択の壁を測る)=====
# 固定scale / pf_mean / 簡易ルール(eval長で鋭さ切替)を比較。
def strat_fixed(w, k): return CACHE[w][k]
def strat_lenrule(w):
    # 評価ゾーンが長い=外挿リスク大→緩いscale(hedge) / 短い→鋭いscale
    n_eval = int(np.isfinite(CACHE[w]["pf_mean"]).sum())  # ev行数の代理
    n_eval = len(truth[w])
    k = "pf_scale_3" if n_eval < 1500 else ("pf_scale_8" if n_eval < 4000 else "pf_scale_12")
    return CACHE[w][k]
def eval_strat(fn):
    e = [fn(w)-truth[w] for w in names]
    eE = [fn(w)-truth[w] for w in EASY]; eH = [fn(w)-truth[w] for w in HARD]
    return rmse(eE), rmse(eH), rmse(e)

print(f"{'strategy':>16s} {'easy':>7s} {'hard':>7s} {'all':>7s}")
for k in ["pf_scale_3","pf_scale_5","pf_scale_8","pf_scale_12","pf_mean"]:
    r = eval_strat(lambda w,k=k: strat_fixed(w,k)); print(f"{('fixed '+k):>16s} {r[0]:7.2f} {r[1]:7.2f} {r[2]:7.2f}")
r = eval_strat(strat_lenrule); print(f"{'len-rule':>16s} {r[0]:7.2f} {r[1]:7.2f} {r[2]:7.2f}")
print(f"{'ORACLE(上限)':>16s} {oe:7.2f} {oh:7.2f} {oa:7.2f}")
print("\n★ 到達可能戦略が oracle にどれだけ届くか。届かなければ選択の壁(=既知の結論)。")

        strategy    easy    hard     all
fixed pf_scale_3    8.55   24.09    9.27
fixed pf_scale_5    8.21   25.00    9.03
fixed pf_scale_8    8.08   25.89    8.98
fixed pf_scale_12    8.01   26.29    8.95
   fixed pf_mean    8.22   24.35    9.00
        len-rule    8.00   26.29    8.94
      ORACLE(上限)    7.09   20.73    7.74

★ 到達可能戦略が oracle にどれだけ届くか。届かなければ選択の壁(=既知の結論)。


In [ ]:
# ===== 評価3: seed数の効き(多いほど正確か? 安定性 vs 精度)=====
# 注意: PFパラメータは32seed regimeで最適。seedを変えると regime がズレうる。
for S in [8, 16, 32, 64]:
    pS = C.r0_params(ns, use_phys=USE_PHYS, n_seeds=S)
    rS = C.run_forward_scales(inps, pS, S, C.DEVICE, 0, CHUNK, SCALES)
    cS = {wid: d for wid, d in zip(names, rS)}
    a8 = rmse([cS[w]["pf_scale_8"]-truth[w] for w in names])
    aM = rmse([cS[w]["pf_mean"]-truth[w] for w in names])
    print(f"seeds={S:3d}: pf_scale_8 all={a8:.3f}  pf_mean all={aM:.3f}")
print("\n★ seed↑で all RMSE が単調改善するか? しなければ『多い=正確ではない(安定性のレバー)』を実証。")